# Embeddings from Scratch — OpenAI-Style Encoder

This notebook builds a **text embedding model** the same way modern APIs like OpenAI `text-embedding-3-small` work at a high level:

1. **Transformer encoder** (bidirectional attention)
2. **Mean pooling** over token vectors
3. **L2 normalization** → unit vectors for cosine search
4. **Contrastive training** on triplets with hard negatives (AllNLI entailment + contradiction)

**Pipeline:** load pairs → tokenize → encode → contrastive loss → similarity / semantic search.

Companion to `llm_from_scratch`: that project uses a *causal* GPT for generation; this uses a *bidirectional* encoder for representation.

In [ ]:
import os
import math
import random
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import tiktoken
from datasets import load_dataset, load_from_disk
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print("Device:", device)

## Architecture (OpenAI-style)

| Step | OpenAI embeddings | This notebook |
|------|-------------------|---------------|
| Tokenizer | `cl100k_base` BPE | `tiktoken` GPT-2 BPE |
| Backbone | Transformer **encoder** | Bidirectional transformer blocks |
| Pooling | Mean over tokens | Masked mean pooling |
| Output | L2-normalized float vector | `F.normalize(..., p=2)` |
| Training | Contrastive + hard-negative triplets | InfoNCE + triplet margin on unit vectors |

Unlike GPT (your `llm_from_scratch` project), there is **no causal mask** — every token can attend to every other token in the sentence.

## 1. Load training data (local)

### Dataset: **AllNLI** (SNLI + MultiNLI)

| | |
|---|---|
| **Source** | [sentence-transformers/all-nli](https://huggingface.co/datasets/sentence-transformers/all-nli) |
| **Local path** | `data/all-nli-pair-class/` (~139 MB) |
| **Training format** | **Triplets** with **hard negatives**: `(anchor, positive, negative)` |
| **Positive** | Entailment hypothesis (`label=0`) — paraphrase of anchor |
| **Hard negative** | Contradiction hypothesis (`label=2`) — same anchor, opposite meaning |

In-batch negatives are easy; hard negatives force the model to separate *subtle* confusions (same topic, wrong fact).

**Download once (if missing):**
```bash
python scripts/download_data.py
```

In [ ]:
from collections import defaultdict

MAX_TRAIN_TRIPLETS = 8_000  # increase toward 100k+ for better quality
ALL_NLI_DIR = "data/all-nli-pair-class"

if not os.path.isdir(ALL_NLI_DIR):
    raise FileNotFoundError(
        f"Missing {ALL_NLI_DIR}. Run: python scripts/download_data.py"
    )


def build_hard_negative_triplets(train_ds, max_triplets: int, seed: int = SEED) -> list[dict]:
    """Group AllNLI by premise → one entailment (pos) + one contradiction (hard neg)."""
    pos_by_premise: dict[str, list[str]] = defaultdict(list)
    neg_by_premise: dict[str, list[str]] = defaultdict(list)

    for row in train_ds:
        premise, hypothesis, label = row["premise"], row["hypothesis"], row["label"]
        if label == 0:
            pos_by_premise[premise].append(hypothesis)
        elif label == 2:
            neg_by_premise[premise].append(hypothesis)

    shared = [p for p in pos_by_premise if p in neg_by_premise]
    rng = random.Random(seed)
    rng.shuffle(shared)

    triplets = []
    for premise in shared:
        positive = rng.choice(pos_by_premise[premise])
        negative = rng.choice(neg_by_premise[premise])
        triplets.append({"anchor": premise, "positive": positive, "negative": negative})
        if len(triplets) >= max_triplets:
            break
    return triplets


all_nli = load_from_disk(ALL_NLI_DIR)
triplets = build_hard_negative_triplets(all_nli["train"], MAX_TRAIN_TRIPLETS, SEED)
rng = random.Random(SEED)
rng.shuffle(triplets)

print(f"Loaded from disk: {ALL_NLI_DIR}")
print(f"Training triplets (hard negatives): {len(triplets):,}")
print("Example triplet:")
print("  anchor:  ", triplets[0]["anchor"])
print("  positive:", triplets[0]["positive"])
print("  negative:", triplets[0]["negative"])

## 2. Tokenize (BPE)

OpenAI uses subword tokenization. We use `tiktoken` (same family as GPT-2 / ChatGPT tokenizers).

Padding tokens are masked out so they do not affect mean pooling.

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
PAD_ID = 0  # unused in GPT-2 vocab; safe pad index
MAX_SEQ_LEN = 128


def encode_text(text: str, max_len: int = MAX_SEQ_LEN) -> tuple[list[int], list[int]]:
    ids = tokenizer.encode(text)[:max_len]
    attn = [1] * len(ids)
    pad = max_len - len(ids)
    if pad:
        ids = ids + [PAD_ID] * pad
        attn = attn + [0] * pad
    return ids, attn


class TripletDataset(Dataset):
    def __init__(self, rows: list[dict]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        a_ids, a_mask = encode_text(row["anchor"])
        p_ids, p_mask = encode_text(row["positive"])
        n_ids, n_mask = encode_text(row["negative"])
        return (
            torch.tensor(a_ids, dtype=torch.long),
            torch.tensor(a_mask, dtype=torch.long),
            torch.tensor(p_ids, dtype=torch.long),
            torch.tensor(p_mask, dtype=torch.long),
            torch.tensor(n_ids, dtype=torch.long),
            torch.tensor(n_mask, dtype=torch.long),
        )


split = int(0.9 * len(triplets))
train_triplets = triplets[:split]
val_triplets = triplets[split:]

train_loader = DataLoader(TripletDataset(train_triplets), batch_size=64, shuffle=True, drop_last=True)
val_loader = DataLoader(TripletDataset(val_triplets), batch_size=64, shuffle=False, drop_last=False)

sample = encode_text("A woman is playing violin.")
print("Vocab size:", tokenizer.n_vocab)
print("Train triplets:", len(train_triplets), "| Val:", len(val_triplets))
print("Encoded ids:", sample[0][:12], "...")

## 3. Transformer encoder (from scratch)

Core components mirror OpenAI's embedding backbone:

- `BidirectionalSelfAttention` — full attention (no future mask)
- `TransformerEncoderBlock` — pre-norm residual block
- `TextEmbeddingModel` — token + position embeddings → blocks → **mean pool** → **L2 norm**

In [ ]:
class EmbeddingConfig:
    def __init__(
        self,
        vocab_size: int,
        n_layer: int = 4,
        n_head: int = 4,
        n_embd: int = 128,
        max_seq_len: int = 128,
    ):
        self.vocab_size = vocab_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.max_seq_len = max_seq_len


def sinusoidal_positions(seq_len: int, d: int) -> torch.Tensor:
    pos = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)
    div = torch.exp(torch.arange(0, d, 2, dtype=torch.float32) * (-math.log(10000.0) / d))
    pe = torch.zeros(seq_len, d)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe


class BidirectionalSelfAttention(nn.Module):
    """Full (non-causal) multi-head self-attention — encoder style."""

    def __init__(self, config: EmbeddingConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.proj = nn.Linear(config.n_embd, config.n_embd)

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        # Mask padded positions: attn_mask (B, T) -> (B, 1, 1, T)
        pad_mask = attn_mask.unsqueeze(1).unsqueeze(2) == 0
        scores = scores.masked_fill(pad_mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class FeedForward(nn.Module):
    def __init__(self, config: EmbeddingConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
        )

    def forward(self, x):
        return self.net(x)


class TransformerEncoderBlock(nn.Module):
    def __init__(self, config: EmbeddingConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = BidirectionalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.mlp = FeedForward(config)

    def forward(self, x, attn_mask):
        x = x + self.attn(self.ln1(x), attn_mask)
        x = x + self.mlp(self.ln2(x))
        return x


def masked_mean_pool(token_vecs: torch.Tensor, attn_mask: torch.Tensor) -> torch.Tensor:
    """OpenAI-style mean pooling: average only non-padding tokens."""
    mask = attn_mask.unsqueeze(-1).float()
    summed = (token_vecs * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-6)
    return summed / counts


class TextEmbeddingModel(nn.Module):
    """Mini OpenAI-style text embedding encoder."""

    def __init__(self, config: EmbeddingConfig):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.blocks = nn.ModuleList([TransformerEncoderBlock(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.register_buffer("pos_encoding", sinusoidal_positions(config.max_seq_len, config.n_embd))
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def encode(self, input_ids: torch.Tensor, attn_mask: torch.Tensor, normalize: bool = True) -> torch.Tensor:
        B, T = input_ids.shape
        x = self.token_embedding(input_ids) + self.pos_encoding[:T].unsqueeze(0)
        for block in self.blocks:
            x = block(x, attn_mask)
        x = self.ln_f(x)
        emb = masked_mean_pool(x, attn_mask)
        if normalize:
            emb = F.normalize(emb, p=2, dim=-1)
        return emb

    def forward(self, input_ids, attn_mask):
        return self.encode(input_ids, attn_mask, normalize=True)


config = EmbeddingConfig(vocab_size=tokenizer.n_vocab, n_layer=4, n_head=4, n_embd=128, max_seq_len=MAX_SEQ_LEN)
model = TextEmbeddingModel(config).to(device)

batch = next(iter(train_loader))
a_ids, a_mask, p_ids, p_mask, n_ids, n_mask = [t.to(device) for t in batch]
with torch.no_grad():
    vec = model(a_ids, a_mask)
print("Embedding shape:", vec.shape)
print("L2 norm (should be ~1):", vec[0].norm().item())

## 4. How this model is optimized for **cosine similarity**

OpenAI embeddings are built for retrieval via **cosine similarity**. Our model matches that in three coupled steps:

### Step 1 — L2 normalization at the output

After mean pooling: $\hat{\mathbf{e}} = \mathbf{e} / \|\mathbf{e}\|_2$. Then $\cos(\hat{\mathbf{e}}_A, \hat{\mathbf{e}}_B) = \hat{\mathbf{e}}_A \cdot \hat{\mathbf{e}}_B$ — dot product alone ranks documents.

### Step 2 — Contrastive loss *is* a cosine objective

With unit vectors, $S_{ij} = \mathbf{a}_i \cdot \mathbf{b}_j = \cos(\mathbf{a}_i, \mathbf{b}_j)$. We maximize cosine on matching pairs (diagonal) and minimize vs. in-batch negatives (InfoNCE / MNRL).

### Step 3 — Hard-negative triplet margin

$$\mathcal{L}_{\text{triplet}} = \max(0,\; \cos(\mathbf{a}, \mathbf{n}) - \cos(\mathbf{a}, \mathbf{p}) + m)$$

Forces $\cos(\text{anchor}, \text{positive}) > \cos(\text{anchor}, \text{hard negative})$ by margin $m$.

**Combined:** $\mathcal{L} = \mathcal{L}_{\text{InfoNCE}} + \lambda \cdot \mathcal{L}_{\text{triplet}}$

In [ ]:
TEMPERATURE = 0.05
TRIPLET_MARGIN = 0.2
HARD_NEGATIVE_WEIGHT = 1.0


def symmetric_contrastive_loss(emb_a: torch.Tensor, emb_b: torch.Tensor, temperature: float = TEMPERATURE) -> torch.Tensor:
    """InfoNCE on cosine scores (dot product of L2-normalized embeddings)."""
    cosine_scores = emb_a @ emb_b.T
    logits = cosine_scores / temperature
    labels = torch.arange(emb_a.size(0), device=emb_a.device)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))


def triplet_margin_loss(
    anchor: torch.Tensor,
    positive: torch.Tensor,
    negative: torch.Tensor,
    margin: float = TRIPLET_MARGIN,
) -> torch.Tensor:
    """Hard negative: push cos(a,n) below cos(a,p) by margin."""
    pos_sim = (anchor * positive).sum(dim=1)
    neg_sim = (anchor * negative).sum(dim=1)
    return F.relu(neg_sim - pos_sim + margin).mean()


def combined_loss(emb_a, emb_p, emb_n):
    return symmetric_contrastive_loss(emb_a, emb_p) + HARD_NEGATIVE_WEIGHT * triplet_margin_loss(emb_a, emb_p, emb_n)


with torch.no_grad():
    emb_a = model(a_ids, a_mask)
    emb_p = model(p_ids, p_mask)
    emb_n = model(n_ids, n_mask)
    print("Smoke-test InfoNCE:", float(symmetric_contrastive_loss(emb_a, emb_p)))
    print("Smoke-test triplet:", float(triplet_margin_loss(emb_a, emb_p, emb_n)))
    print("Smoke-test combined:", float(combined_loss(emb_a, emb_p, emb_n)))
    print("Mean cos(anchor, positive):", float((emb_a * emb_p).sum(1).mean()))
    print("Mean cos(anchor, negative):", float((emb_a * emb_n).sum(1).mean()))

## 4b. Snapshot cosine scores **before** training

Fixed demo pairs — we'll replot these after epoch 5 to visualize learning.

In [ ]:
DEMO_PAIRS = [
    ("A dog is running in the park.", "A canine is playing outside.", "similar"),
    ("A dog is running in the park.", "The stock market crashed yesterday.", "dissimilar"),
    ("She is playing violin.", "A woman performs music on a violin.", "similar"),
    ("Paris is the capital of France.", "Berlin is the capital of Germany.", "similar"),
    ("The weather is sunny today.", "Quantum physics explains subatomic particles.", "dissimilar"),
    ("He is cooking pasta in the kitchen.", "A man prepares Italian food at home.", "similar"),
    ("The basketball team won the game.", "Neural networks use gradient descent.", "dissimilar"),
    ("Two students are studying for an exam.", "A pair of pupils reviews course material.", "similar"),
]


@torch.no_grad()
def score_demo_pairs(m: nn.Module) -> list[dict]:
    m.eval()
    results = []
    for text_a, text_b, kind in DEMO_PAIRS:
        ids_a, mask_a = encode_text(text_a)
        ids_b, mask_b = encode_text(text_b)
        t_a = torch.tensor([ids_a], dtype=torch.long, device=device)
        m_a = torch.tensor([mask_a], dtype=torch.long, device=device)
        t_b = torch.tensor([ids_b], dtype=torch.long, device=device)
        m_b = torch.tensor([mask_b], dtype=torch.long, device=device)
        va = m(t_a, m_a)[0].cpu()
        vb = m(t_b, m_b)[0].cpu()
        results.append({"kind": kind, "cosine": float(torch.dot(va, vb)), "a": text_a, "b": text_b})
    return results


def summarize_demo(scores: list[dict], label: str) -> None:
    sim = [s["cosine"] for s in scores if s["kind"] == "similar"]
    dis = [s["cosine"] for s in scores if s["kind"] == "dissimilar"]
    print(f"{label}")
    print(f"  similar pairs    — mean cosine: {np.mean(sim):.3f}")
    print(f"  dissimilar pairs — mean cosine: {np.mean(dis):.3f}")
    print(f"  gap (sim - dis)  — {np.mean(sim) - np.mean(dis):.3f}")


cosines_before = score_demo_pairs(model)
summarize_demo(cosines_before, "Before training (epoch 0):")

## 5. Train

In [ ]:
EPOCHS = 5
LEARNING_RATE = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
train_history, val_history = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for a_ids, a_mask, p_ids, p_mask, n_ids, n_mask in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        a_ids, a_mask = a_ids.to(device), a_mask.to(device)
        p_ids, p_mask = p_ids.to(device), p_mask.to(device)
        n_ids, n_mask = n_ids.to(device), n_mask.to(device)

        optimizer.zero_grad()
        emb_a = model(a_ids, a_mask)
        emb_p = model(p_ids, p_mask)
        emb_n = model(n_ids, n_mask)
        loss = combined_loss(emb_a, emb_p, emb_n)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running += loss.item()

    train_loss = running / len(train_loader)
    train_history.append(train_loss)

    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for a_ids, a_mask, p_ids, p_mask, n_ids, n_mask in val_loader:
            a_ids, a_mask = a_ids.to(device), a_mask.to(device)
            p_ids, p_mask = p_ids.to(device), p_mask.to(device)
            n_ids, n_mask = n_ids.to(device), n_mask.to(device)
            emb_a = model(a_ids, a_mask)
            emb_p = model(p_ids, p_mask)
            emb_n = model(n_ids, n_mask)
            val_running += combined_loss(emb_a, emb_p, emb_n).item()
    val_loss = val_running / max(1, len(val_loader))
    val_history.append(val_loss)

    print(f"Epoch {epoch} — train: {train_loss:.4f}, val: {val_loss:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(train_history, marker="o", label="train")
plt.plot(val_history, marker="o", label="val")
plt.xlabel("Epoch")
plt.ylabel("Contrastive loss")
plt.title("OpenAI-style contrastive training")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5b. Before vs after training — cosine similarity plot

Same demo pairs as epoch 0. We expect:
- **Similar** pairs → higher cosine after training
- **Dissimilar** pairs → lower cosine after training
- **Gap** (mean similar − mean dissimilar) → increases

In [ ]:
cosines_after = score_demo_pairs(model)
summarize_demo(cosines_after, f"After training (epoch {EPOCHS}):")

before_vals = [s["cosine"] for s in cosines_before]
after_vals = [s["cosine"] for s in cosines_after]
kinds = [s["kind"] for s in cosines_before]
x = np.arange(len(DEMO_PAIRS))

fig, ax = plt.subplots(figsize=(12, 5))
width = 0.35
ax.bar(x - width / 2, before_vals, width, label="before (epoch 0)", color="#94a3b8")
ax.bar(x + width / 2, after_vals, width, label=f"after (epoch {EPOCHS})", color="#2563eb")
ax.set_xticks(x)
ax.set_xticklabels([f"{i+1}\n({k})" for i, k in enumerate(kinds)], fontsize=9)
ax.set_ylabel("Cosine similarity")
ax.set_xlabel("Demo pair")
ax.set_title("Before vs after training on fixed sentence pairs")
ax.axhline(0, color="black", linewidth=0.5)
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Summary gap chart
def mean_by_kind(scores, kind):
    return np.mean([s["cosine"] for s in scores if s["kind"] == kind])

labels = ["similar", "dissimilar", "gap (sim − dis)"]
before_summary = [mean_by_kind(cosines_before, "similar"), mean_by_kind(cosines_before, "dissimilar"),
                  mean_by_kind(cosines_before, "similar") - mean_by_kind(cosines_before, "dissimilar")]
after_summary = [mean_by_kind(cosines_after, "similar"), mean_by_kind(cosines_after, "dissimilar"),
                 mean_by_kind(cosines_after, "similar") - mean_by_kind(cosines_after, "dissimilar")]

x2 = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x2 - width / 2, before_summary, width, label="before", color="#94a3b8")
ax.bar(x2 + width / 2, after_summary, width, label="after", color="#2563eb")
ax.set_xticks(x2)
ax.set_xticklabels(labels)
ax.set_ylabel("Mean cosine")
ax.set_title("Aggregate separation: similar vs dissimilar")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Evaluate — sentence similarity

Because vectors are L2-normalized, **cosine similarity = dot product**.

In [ ]:
@torch.no_grad()
def embed_text(text: str) -> torch.Tensor:
    ids, mask = encode_text(text)
    t_ids = torch.tensor([ids], dtype=torch.long, device=device)
    t_mask = torch.tensor([mask], dtype=torch.long, device=device)
    return model(t_ids, t_mask)[0].cpu()


def cosine_sim(a: str, b: str) -> float:
    va, vb = embed_text(a), embed_text(b)
    return float(torch.dot(va, vb))


pairs = [
    ("A dog is running in the park.", "A canine is playing outside.", "similar"),
    ("The stock market fell today.", "Investors lost money on equities.", "similar"),
    ("A dog is running in the park.", "The quantum computer solved the equation.", "dissimilar"),
    ("She is playing violin.", "A woman performs music on a violin.", "similar"),
]

print(f"{'Sentence A':<42} {'Sentence B':<42} {'cosine':>8}")
print("-" * 95)
for a, b, _ in pairs:
    print(f"{a[:40]:<42} {b[:40]:<42} {cosine_sim(a, b):8.3f}")

### 6b. Benchmark on STS (local eval set)

**STS Benchmark** (`data/stsbenchmark/`) has human similarity scores 0–5. We check whether model cosine scores correlate with human judgments (Spearman ρ).

In [ ]:
from scipy.stats import spearmanr

STS_DIR = "data/stsbenchmark"
if os.path.isdir(STS_DIR):
    sts = load_from_disk(STS_DIR)["test"]
    sts = sts.select(range(min(300, len(sts))))  # quick eval subset

    human_scores, model_scores = [], []
    for row in sts:
        s1, s2, gold = row["sentence1"], row["sentence2"], row["score"]
        human_scores.append(gold)
        model_scores.append(cosine_sim(s1, s2))

    rho, pval = spearmanr(human_scores, model_scores)
    print(f"STS test subset: {len(sts)} pairs")
    print(f"Spearman correlation (human vs model cosine): {rho:.3f}  (p={pval:.2e})")
    print("Higher ρ => better alignment with human similarity judgments")
else:
    print(f"STS not found at {STS_DIR}. Run: python scripts/download_data.py")

## 7. Semantic search

This is exactly how RAG retrieval works: embed query and documents, rank by dot product.

In [ ]:
corpus = [
    "Machine learning models learn patterns from data.",
    "Paris is the capital city of France.",
    "Neural networks use layers of connected neurons.",
    "The basketball team won the championship game.",
    "Embeddings map text into dense vector space for search.",
    "Italian pasta is popular around the world.",
    "Transformers revolutionized natural language processing.",
    "The Pacific Ocean is the largest ocean on Earth.",
]

corpus_vecs = torch.stack([embed_text(doc) for doc in corpus])


def semantic_search(query: str, top_k: int = 3):
    q = embed_text(query)
    sims = corpus_vecs @ q
    top = torch.topk(sims, k=top_k)
    print(f"Query: {query}\n")
    for rank, (score, idx) in enumerate(zip(top.values.tolist(), top.indices.tolist()), 1):
        print(f"{rank}. {score:.3f} — {corpus[idx]}")


semantic_search("How do language models represent text?")
print()
semantic_search("European capital cities")

## 8. Visualize with t-SNE

Plot sentence clusters — great visual for LinkedIn.

In [ ]:
probe_sentences = [
    "Dogs run in the park.",
    "A puppy plays outside.",
    "Cats sleep on the sofa.",
    "Stock prices dropped sharply.",
    "The market declined today.",
    "She studies computer science.",
    "He researches machine learning.",
    "Pizza is a popular Italian food.",
    "Pasta dishes come from Italy.",
]

probe_vecs = torch.stack([embed_text(s) for s in probe_sentences]).numpy()
coords = TSNE(n_components=2, perplexity=3, random_state=SEED, init="pca").fit_transform(probe_vecs)

plt.figure(figsize=(10, 7))
plt.scatter(coords[:, 0], coords[:, 1], s=80, alpha=0.85)
for i, sent in enumerate(probe_sentences):
    label = sent[:28] + ("..." if len(sent) > 28 else "")
    plt.annotate(label, (coords[i, 0], coords[i, 1]), fontsize=9, xytext=(4, 4), textcoords="offset points")
plt.title("OpenAI-style sentence embeddings (t-SNE)")
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 9. Save checkpoint

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
checkpoint_path = "checkpoints/embedding_encoder_scratch.pt"

torch.save(
    {
        "model_state": model.state_dict(),
        "config": vars(config),
        "tokenizer": "gpt2",
        "training": {
            "epochs": EPOCHS,
            "temperature": TEMPERATURE,
            "max_train_triplets": MAX_TRAIN_TRIPLETS,
            "triplet_margin": TRIPLET_MARGIN,
            "hard_negative_weight": HARD_NEGATIVE_WEIGHT,
        },
    },
    checkpoint_path,
)
print("Saved:", checkpoint_path)

## Next steps

1. **Scale up:** `MAX_TRAIN_TRIPLETS=100_000`, `n_embd=256`, `n_layer=6`
2. **Mini RAG notebook:** chunk WikiText → embed → retrieve (pairs with `llm_from_scratch`)
3. **Compare to OpenAI API:** embed the same sentences with `text-embedding-3-small`
4. **Matryoshka dims:** truncate vectors to 64/32 dims, re-normalize, test retrieval

---

**GitHub:** `github.com/debanshu005/embedding_model_from_scratch`